## Setup

In [ ]:
import sys
sys.path.insert(0, 'E:\Code\Music')

In [ ]:
from pathlib import Path
from PDMX.reading.music import MusicRender
import random
import pandas as pd
from collections import Counter
import torch.nn as nn
import torch.nn.functional as F
import torch
import pretty_midi
from torch.utils.data import TensorDataset, DataLoader
import scipy.io.wavfile as wav
import IPython.display as ipd
import fluidsynth
from tqdm import tqdm

In [ ]:
PDMX_dir = Path('E:\Code\music\dataset')

## Data Processing

In [ ]:
def extract_note_sequences(json_file: str) -> list[list]:
    note_sequences = []
    music = MusicRender()
    music.load(json_file)
      
    for track in music.tracks:
        notes = [note.pitch for note in track.notes]
        note_sequences.append(notes)
    
    return note_sequences
    

In [ ]:
json_root = PDMX_dir / 'data'
json_files = [str(file) for file in json_root.rglob('*.json')]

In [ ]:
all_sequences = []

for path in (random.sample(json_files, k=1000)):
    notes = extract_note_sequences(path)
    all_sequences.extend(seq for seq in notes)


In [ ]:
all_notes = [note for seq in all_sequences for note in seq]
note_counts = Counter(all_notes)

vocab = sorted(note_counts.keys())
idx_to_note = {idx : note for idx, note in enumerate(vocab)}
note_to_idx = {note : idx for idx, note in idx_to_note.items()}

assert len(vocab) == len(idx_to_note) == len(note_to_idx)

VOCAB_SIZE = len(vocab)

In [ ]:
encoded_sequences = [[note_to_idx[pitch] for pitch in sequence] for sequence in all_sequences]
assert len(encoded_sequences) == len(all_sequences)

In [ ]:
sequence_length = 64

x_data = []
y_data = []
for sequence in encoded_sequences:
    for i in range(len(sequence) - sequence_length):
        x_data.append(sequence[i:i+sequence_length])
        y_data.append(sequence[i+1:i+1+sequence_length])

assert len(x_data) == len(y_data)

## Model and DataLoader

Convert `x_data` / `y_data` to tensors and wrap them in a `DataLoader`. `TensorDataset` pairs them by index; the `DataLoader` handles batching and shuffling each epoch.

In [ ]:
class MusicTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, max_seq_len=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embedding = nn.Embedding(max_seq_len, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4, dim_feedforward=256, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)
        self.out = nn.Linear(embed_dim, vocab_size)
        
    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        x = self.embedding(x) + self.pos_embedding(positions)
        mask = nn.Transformer.generate_square_subsequent_mask(x.size(1), device=x.device)
        x = self.transformer(x, mask=mask, is_causal=True)
        return self.out(x)

In [ ]:
x_tensor = torch.tensor(x_data, dtype=torch.long)
y_tensor = torch.tensor(y_data, dtype=torch.long)

dataset = TensorDataset(x_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

print(f"Dataset size : {len(dataset):,} sequences")
print(f"Batches/epoch: {len(loader):,}")

## Loss, Optimizer & Scheduler

- **Device**: moves all computation to GPU if available; otherwise CPU.
- **CrossEntropyLoss**: fused log-softmax + NLL for stable 95-class prediction at every position.
- **AdamW** (`lr=3e-4`, `weight_decay=1e-2`): adaptive per-parameter LR with correctly decoupled weight decay.
- **CosineAnnealingLR**: smoothly decays LR from `3e-4` to ~0 over `T_max` epochs so the model settles precisely near a minimum.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

model = MusicTransformer(VOCAB_SIZE).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

## Training Loop

Each step:
1. Zero gradients — clear accumulated gradients from the previous step.
2. Forward pass — model produces `(batch, 64, vocab_size)` logits.
3. Reshape — flatten to `(batch×64, vocab_size)` and `(batch×64,)` for CrossEntropyLoss.
4. Backward — compute gradients via autograd.
5. Clip — rescale the global gradient norm to ≤ 1.0 to prevent exploding gradients.
6. Step — update weights; then step the LR scheduler once per epoch.

In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for x_batch, y_batch in tqdm(loader):
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(x_batch)                         # (batch, 64, vocab_size)
        loss = criterion(
            logits.view(-1, VOCAB_SIZE),                # (batch*64, vocab_size)
            y_batch.view(-1)                            # (batch*64,)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()
    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1:>2}/{EPOCHS}  loss: {avg_loss:.4f}  lr: {scheduler.get_last_lr()[0]:.2e}")

## Save Model

In [ ]:
# saving the model
# torch.save(model.state_dict(), 'music_transformer.pth')

## Load Model

In [ ]:
model = MusicTransformer(90)
model.load_state_dict(torch.load('music_transformer.pth'))
model.to(device)
model.eval()

## Generating New Music

In [ ]:
def notes_to_midi(note_sequence, output_path='output.mid', tempo=120):
    pm = pretty_midi.PrettyMIDI(initial_tempo=tempo)
    instrument = pretty_midi.Instrument(program=0)
    
    step = 60.0 / tempo
    for i, pitch in enumerate(note_sequence):
        note = pretty_midi.Note(velocity=80, pitch=pitch, start=i*step, end=(i+1)*step)
        instrument.notes.append(note)
    
    pm.instruments.append(instrument)
    pm.write(output_path)
    return pm

In [ ]:
# select a random sequence from the training data size(1, 64)
seed_idx = torch.randint(0, len(x_data), (1,)).item()
seed = torch.tensor(x_data[seed_idx]).unsqueeze(0).to(device)

In [ ]:
temp = 1.0
seq_length = seed.shape[1]
for _ in range(seq_length):
    output = model(seed)
    logits = torch.softmax(output[0, -1, :] / temp, -1)
    sample_idx = torch.multinomial(logits, 1).unsqueeze(0)
    seed = torch.cat([seed, sample_idx], dim=1)[:, 1:]

In [ ]:
seed = seed.flatten()
note_indices = seed.tolist()
note_sequence = [idx_to_note[idx] for idx in note_indices]
pm = notes_to_midi(note_sequence)


In [73]:
sf2_path = r'c:\Users\James\Music\Samples\GeneralUser_GS_v2.0.3--doc_r6\GeneralUser-GS\GeneralUser-GS.sf2'
audio = pm.fluidsynth(fs=44100, sf2_path=sf2_path)
audio_int16 = (audio * 32767).astype('int16')
wav.write('output.wav', rate=44100, data=audio_int16)

In [75]:
ipd.Audio('output.wav', embed=True)